In [ ]:
!pip install evaluate rouge_score sacrebleu bert_score scikit-learn pandas

# Inference & Evaluation Matrix

In [ ]:
import json
import re
import pandas as pd
import numpy as np
import warnings
from sklearn.metrics import accuracy_score, f1_score, classification_report

# --- NLG Metric Imports ---
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_scorer

warnings.filterwarnings('ignore') # Suppress bert-score warnings

# ==========================================
# 1. VERDICT EXTRACTION
# ==========================================
def extract_verdict(text):
    tl = str(text).strip().lower()
    matches = re.findall(r'verdict:\s*(supported|true|refuted|false)', tl)
    if matches: return 1 if matches[-1] in ('supported', 'true') else 0
    if re.match(r'the claim is (true|correct|supported)', tl): return 1
    if re.match(r'the claim is (false|incorrect|wrong)', tl):  return 0
    false_n = len(re.findall(r'\b(false|refuted)\b', tl))
    true_n  = len(re.findall(r'\b(true|supported)\b', tl))
    if false_n > true_n: return 0
    if true_n > false_n: return 1
    return -1

# ==========================================
# 2. CHART TYPE MAPPING
# ==========================================
def map_spec_to_fixed_label(spec, csv_label):
    topo = spec.get("topo", {})
    t = topo.get("type", "")
    sub = topo.get("sub", "")

    if isinstance(t, list): types = [str(x).lower() for x in t]
    elif isinstance(t, str): types = [t.lower()]
    else: types = []
    sub = sub.lower() if isinstance(sub, str) else ""

    if (not types or types == [""]) and sub == "": return csv_label
    if "bar+line" in types or ("bar" in types and "line" in types): return "bar+line"
    if "area+line" in types: return "area"
    if "pie" in types: return "pie_chart"
    if "circle" in types and sub == "donut": return "pie_chart"
    if "bar" in types: return "barchart_horizontal" if sub == "horizontal" else "barchart_vertical"
    if "line" in types: return "line_chart"
    if "histogram" in types: return "histogram"
    if "area" in types: return "area"
    if "scatter" in types: return "scatter_plot"

    return csv_label

# ==========================================
# 3. NLG METRICS CALCULATION
# ==========================================
def compute_nlg_metrics(preds, refs):
    print("\nCalculating NLG Metrics (BLEU, ROUGE-L, BERTScore)... This may take a moment.")
    
    # 1. BLEU Score
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    
    # 2. ROUGE-L Score
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(r, p)['rougeL'].fmeasure for r, p in zip(refs, preds)]
    rouge_l = (sum(rouge_scores) / len(rouge_scores)) * 100
    
    # 3. BERTScore
    # Uses DistilRoBERTa by default which is fast and standard for English evaluation
    _, _, F1 = bert_scorer(preds, refs, lang="en", verbose=False)
    bert_f1 = F1.mean().item() * 100
    
    return bleu, rouge_l, bert_f1

# ==========================================
# 4. DATA LOADING & PROCESSING
# ==========================================
def evaluate_model(test1_path, test2_path):
    print("Loading datasets and extracting predictions...")
    
    def process_file(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except FileNotFoundError:
            print(f"Warning: {filepath} not found.")
            return pd.DataFrame(), [], []

        rows, preds_text, refs_text = [], [], []
        
        for item in data:
            gt_label = 1 if str(item.get('label', '')).strip().lower() in ('true', '1', 'supported') else 0
            rationale = item.get("generated_rationale", "")
            gt_explanation = item.get("explanation", "")
            
            pred = extract_verdict(rationale)
            if pred == -1: pred = 1 - gt_label 
            
            spec_dict = {}
            raw_spec = item.get("extended_spec") or item.get("spec") or "{}"
            if isinstance(raw_spec, str):
                try: spec_dict = json.loads(raw_spec)
                except: pass
            elif isinstance(raw_spec, dict): spec_dict = raw_spec
                
            orig_chart_type = item.get("chart_type", item.get("Chart Type", "unknown"))
            mapped_type = map_spec_to_fixed_label(spec_dict, orig_chart_type)
            reasoning_cat = item.get("reasoning_category", "Unknown")

            rows.append({
                "gt": gt_label, "pred": pred,
                "chart_type": mapped_type, "reasoning_category": reasoning_cat
            })
            
            # Store text for NLG evaluation (only if GT exists)
            if gt_explanation.strip():
                preds_text.append(rationale)
                refs_text.append(gt_explanation)
            
        return pd.DataFrame(rows), preds_text, refs_text

    df1, p_text1, r_text1 = process_file(test1_path)
    df2, p_text2, r_text2 = process_file(test2_path)

    if df1.empty or df2.empty:
        print("Evaluation aborted due to missing files.")
        return

    # --- Compute Classification Metrics ---
    t1_acc = accuracy_score(df1["gt"], df1["pred"]) * 100
    t1_f1 = f1_score(df1["gt"], df1["pred"], average='macro') * 100
    t2_acc = accuracy_score(df2["gt"], df2["pred"]) * 100
    t2_f1 = f1_score(df2["gt"], df2["pred"], average='macro') * 100
    avg_acc = (t1_acc + t2_acc) / 2
    nli_baseline = 73.8

    # --- Compute Text Metrics (Combined) ---
    all_preds = p_text1 + p_text2
    all_refs = r_text1 + r_text2
    bleu, rouge_l, bert_f1 = compute_nlg_metrics(all_preds, all_refs)

    # ==========================================
    # 5. PRINTING THE MATRIX
    # ==========================================
    print("\n" + "=" * 115)
    print(" CHARTCHECK EVALUATION MATRIX — Qwen Base Model (Zero-Shot) vs Ours")
    print("=" * 115)
    print("                         Model    Task  Test1_Acc  Test1_F1  Test2_Acc  Test2_F1  Avg_Acc  BLEU ROUGE-L BERTScore")
    print("          DePlot-DeBERTa-class        C       75.0      75.0       72.5      72.5     73.8     -       -         -")
    print("  DePlot-FlanT5-finetune-multi        M       65.7      65.7       65.9      65.8     65.8  17.3    46.3      91.5")
    print("         MatCha-finetune-multi        M       59.4      59.1       61.1      60.9     60.2  17.1    37.3      67.8")
    print("             GPT4V (Zero-Shot)        M       73.8      73.5       72.0      71.3     72.9  10.0    32.3      89.1")
    print("     Qwen2.5-VL-7B (Zero-Shot)        M       82.3      82.2       85.0      85.0     83.7   5.1    32.7      89.5")
    print(f"            Qwen-3B-DPO (Ours)        M       {t1_acc:.1f}      {t1_f1:.1f}       {t2_acc:.1f}      {t2_f1:.1f}     {avg_acc:.1f}  {bleu:>4.1f}    {rouge_l:>4.1f}      {bert_f1:>4.1f}")
    print("=" * 115)
    
    print(f"\nAvg accuracy vs NLI-DeBERTa baseline : {avg_acc - nli_baseline:+.1f}% (above)")
    print(f"\nTest 1  →  Acc: {t1_acc:.1f}%   Macro-F1: {t1_f1:.1f}")
    print(f"Test 2  →  Acc: {t2_acc:.1f}%   Macro-F1: {t2_f1:.1f}")
    print(f"Average →  Acc: {avg_acc:.1f}%\n")

    # ==========================================
    # 6. PER-CLASS & PER-CHART REPORTS
    # ==========================================
    print("Test 1 — Per-class report:")
    print(classification_report(df1["gt"], df1["pred"], target_names=["REFUTED", "SUPPORTED"]))

    def print_chart_acc(df, name):
        print(f"\nPer-Chart-Type Accuracy ({name})")
        print("-" * 60)
        results = []
        for ctype, group in df.groupby("chart_type"):
            acc = accuracy_score(group["gt"], group["pred"]) * 100
            results.append((ctype, acc, len(group)))
            
        results.sort(key=lambda x: -x[2]) 
        for ctype, acc, count in results:
            print(f"{ctype:<30} {acc:>6.1f}%    {count}")
        print("-" * 60)
        print(f"Total samples accounted for: {len(df)} / {len(df)}")

    print_chart_acc(df1, "Test 1")
    print_chart_acc(df2, "Test 2")

    # ==========================================
    # 7. REASONING CATEGORY
    # ==========================================
    print("\nReasoning Category Performance (Combined Tests)")
    print("-" * 100)
    print(f"{'Reasoning Category':<40} {'Count':<10} {'Generative CoT Accuracy':<25}")
    
    df_all = pd.concat([df1, df2])
    cat_results = []
    for cat, group in df_all.groupby("reasoning_category"):
        if cat == "Unknown": continue
        acc = accuracy_score(group["gt"], group["pred"]) * 100
        cat_results.append((cat, acc, len(group)))
        
    cat_results.sort(key=lambda x: -x[2])
    for cat, acc, count in cat_results:
        print(f"{cat:<40} {count:<10} {acc:>6.1f}%")
    print("-" * 100)

# Run the evaluation!
evaluate_model("test_1_with_DPO_rationales.json", "test_2_with_DPO_rationales.json")